# Sinusoidal Positional Encoding

源码导航：[core/position/sinusoidal.py](../../../core/position/sinusoidal.py) 中的 `SinusoidalPositionalEncoding`。

Vaswani et al. (2017) 在 *Attention Is All You Need* 中提出正弦固定位置编码。与 GPT-2 的可学习位置向量不同，正弦编码通过预定义的三角函数生成位置向量，不引入任何可训练参数，同时允许模型通过线性组合学习到相对位置的表示。

### 1. 理论推导

对于位置 $pos$ 和维度索引 $i \in [0, d_{\text{model}})$，正弦编码定义为：

$$\text{PE}(pos, 2i) = \sin\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)$$

$$\text{PE}(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)$$

**核心设计思想**：
- 低维度（小 $i$）对应低频（大波长），编码宏观位置变化；
- 高维度（大 $i$）对应高频（小波长），编码微观位置变化；
- 三角函数使模型能够通过 $\sin(a+b)$ 和 $\cos(a+b)$ 的线性组合学习到相对位置 $pos + k$ 的表示。

**与 Learned PE 的对比**：

| 属性 | Learned PE | Sinusoidal PE |
|---|---|---|
| 参数 | 可学习 ($T_{\max} \times d$) | 固定（无参数） |
| 外推性 | 超过 $T_{\max}$ 行为未定义 | 理论上可外推到任意长度 |
| 相对位置 | 隐式学习 | 显式编码于三角函数周期中 |
| 训练稳定性 | 需小心初始化 | 完全确定，无需优化 |

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.position.sinusoidal import SinusoidalPositionalEncoding

### 2. 形状与 dtype 检查

In [ ]:
torch.manual_seed(0)
pe = SinusoidalPositionalEncoding(d_model=64, max_seq_len=128)

x = pe(seq_len=16)
print("PE shape:", tuple(x.shape))   # (16, 64)
print("PE dtype:", x.dtype)           # torch.float32
print("PE device:", x.device)         # cpu
assert x.shape == (16, 64), "形状必须匹配 (seq_len, d_model)！"
assert not any(p.requires_grad for p in pe.parameters()), "正弦编码不应有任何可学习参数！"
print("参数数量:", sum(p.numel() for p in pe.parameters()))  # 应为 0

### 3. 频率分布验证

验证不同维度的波长是否按预期递减：维度越低，波长越长（频率越低）。

In [ ]:
import math

d_model = 64
base = 10000.0
half = d_model // 2

wavelengths = []
for i in range(half):
    freq = 1.0 / (base ** (2.0 * i / d_model))
    wavelength = 2 * math.pi / freq
    wavelengths.append(wavelength)

print(f"维度 0 波长: {wavelengths[0]:.2f}（极低频，宏观位置）")
print(f"维度 {half-1} 波长: {wavelengths[-1]:.6f}（极高频，微观位置）")
print(f"波长比值: {wavelengths[0] / wavelengths[-1]:.2f}")
assert wavelengths[0] > wavelengths[-1], '波长应随维度增加而递减'

### 4. 相对位置线性组合特性

正弦编码的关键优势：对于固定的偏移 $k$，$\text{PE}(pos + k)$ 可以表示为 $\text{PE}(pos)$ 的线性函数。这意味着模型可以通过学习简单的权重就能感知相对距离。

In [ ]:
pe = SinusoidalPositionalEncoding(d_model=64, max_seq_len=128)
pos_10 = pe(seq_len=128)[10]   # 位置 10
pos_15 = pe(seq_len=128)[15]   # 位置 15
pos_20 = pe(seq_len=128)[20]   # 位置 20

diff_1 = (pos_15 - pos_10).abs().mean().item()
diff_2 = (pos_20 - pos_15).abs().mean().item()

print(f"|PE(15) - PE(10)| 均值: {diff_1:.6f}")
print(f"|PE(20) - PE(15)| 均值: {diff_2:.6f}")
print(f"两者比值: {diff_1 / diff_2:.4f}（应接近 1.0，证明相同相对偏移产生相同变化模式）")
assert abs(diff_1 - diff_2) < 1e-4, '相同相对偏移的变化幅度应相等'

### 5. 超长序列动态扩展

当请求的序列长度超过预缓存的 `max_seq_len` 时，模块会动态重新计算更长的编码表。

In [ ]:
pe_small = SinusoidalPositionalEncoding(d_model=32, max_seq_len=64)
print("初始缓存长度:", pe_small.pe.size(0))

x_long = pe_small(seq_len=100)
print("动态扩展后长度:", pe_small.pe.size(0))
print("输出形状:", tuple(x_long.shape))
assert x_long.shape == (100, 32), "动态扩展后形状应正确"

### 6. 源码精讲

```python
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int = 8192, base: float = 1e4):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.base = base
        # 预计算并注册为非持久化 buffer，随 .to(device) 迁移，但不存入 state_dict
        pe = self._build_pe(max_seq_len)
        self.register_buffer("pe", pe, persistent=False)

    def _build_pe(self, seq_len: int) -> torch.Tensor:
        position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)  # (seq_len, 1)
        div_term = torch.exp(
            torch.arange(0, self.d_model, 2, dtype=torch.float32)
            * (-math.log(self.base) / self.d_model)
        )  # (d_model // 2,)
        pe = torch.zeros(seq_len, self.d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, seq_len: int, device=None) -> torch.Tensor:
        if seq_len > self.pe.size(0):
            self.pe = self._build_pe(seq_len).to(self.pe.device)
        pe = self.pe[:seq_len]
        if device is not None:
            pe = pe.to(device)
        return pe
```

关键设计点：
- `register_buffer(..., persistent=False)` 使编码随设备自动迁移，但不参与 state_dict 保存（固定值无需持久化）。
- 奇偶维度分别使用 sin/cos，形成相位差 90° 的正交基，增强表达能力。
- 动态扩展机制允许推理时处理超过训练长度的序列。

---

## 延伸阅读与参考资料

### 核心论文
- **Attention Is All You Need**: Vaswani et al., 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)

### 相关讨论
- **BERT**: Devlin et al., 2018. 使用可学习 PE 替代正弦编码，引发关于两种方案优劣的长期讨论。
- **Transformer-XL**: Dai et al., 2019. 引入相对位置编码，进一步扩展了位置表示的边界。